In [2]:
os.chdir("/home/info-sec-lab/BTP/hybrid")

In [1]:
import os
import math
import numpy as np
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForMaskedLM

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "microsoft/codebert-base-mlm"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name).to(device)
model.eval()

Some weights of the model checkpoint at microsoft/codebert-base-mlm were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


RobertaForMaskedLM(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNor

In [4]:
def load_text_data(folder):
    texts, labels = [], []

    for label_name, label in [("Label_0", 0), ("Label_1", 1)]:
        subfolder = os.path.join(folder, label_name)

        for file in sorted(os.listdir(subfolder)):
            if file.endswith(".txt"):
                with open(os.path.join(subfolder, file), "r", encoding="utf-8") as f:
                    texts.append(f.read())
                labels.append(label)

    return texts, np.array(labels)

In [5]:
def compute_metrics(text, max_len=256):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_len
    )

    input_ids = inputs["input_ids"].to(device)
    seq_len = input_ids.size(1)

    logits_list = []
    target_ids = []

    # Mask each token (skip special tokens)
    for i in range(1, seq_len - 1):
        masked = input_ids.clone()
        masked[0, i] = tokenizer.mask_token_id

        with torch.no_grad():
            outputs = model(masked)

        logits = outputs.logits[0, i].detach().cpu()  # 🔥 move to CPU
        logits_list.append(logits)

        target_ids.append(input_ids[0, i].detach().cpu())

    # Stack once (efficient)
    logits_tensor = torch.stack(logits_list)   # [T, V]
    targets = torch.tensor(target_ids)         # [T]

    # =========================
    # Compute metrics
    # =========================

    probs = torch.softmax(logits_tensor, dim=-1)
    log_probs = torch.log(probs)

    # (1) Log-Rank
    sorted_ids = torch.argsort(logits_tensor, dim=-1, descending=True)
    ranks = (sorted_ids == targets.unsqueeze(1)).nonzero()[:, 1] + 1
    log_rank = torch.log(ranks.float()).mean().item()

    # (2) Entropy
    entropy = (-(probs * log_probs).sum(dim=-1)).mean().item()

    # (3–5) GLTR Top-k ratios
    t = targets.unsqueeze(1)

    top10 = torch.topk(logits_tensor, 10, dim=-1).indices
    top100 = torch.topk(logits_tensor, 100, dim=-1).indices
    top1000 = torch.topk(logits_tensor, 1000, dim=-1).indices

    top10_ratio = (top10 == t).any(dim=1).float().mean().item()
    top100_ratio = (top100 == t).any(dim=1).float().mean().item()
    top1000_ratio = (top1000 == t).any(dim=1).float().mean().item()

    # 🔥 free memory
    del logits_tensor, probs, log_probs
    torch.cuda.empty_cache()

    return [
        log_rank,
        entropy,
        top10_ratio,
        top100_ratio,
        top1000_ratio
    ]

In [6]:


def compute_dataset(texts, name="Dataset"):
    results = []

    for text in tqdm(texts, desc=f"Metrics ({name})"):
        results.append(compute_metrics(text))  # your optimized function

    return np.array(results)

In [7]:
os.chdir("/home/info-sec-lab/BTP/hybrid")

In [8]:
# Define all dataset paths
dataset_paths = {
    "test": "./Text_Files/test"
}

# Process each dataset
for dataset_type, DATASET_PATH in dataset_paths.items():
    texts, labels = load_text_data(DATASET_PATH)
    metrics = compute_dataset(texts, f"{dataset_type.capitalize()} Dataset")
    
    np.savez(
        f"./metrics/metrics_{dataset_type}.npz",
        metrics=metrics,
        labels=labels
    )

Metrics (Test Dataset): 100%|██████████| 20586/20586 [7:20:43<00:00,  1.28s/it]  
